In [1]:
import pymupdf4llm
loaded_document=pymupdf4llm.to_markdown("edited structured tables.pdf")


Processing edited structured tables.pdf...
[                                        ] (0/13[                                        ] (  1/13[                                        ] (  2/13[                                        ] (  3/138[=                                       ] (  4/138[=                                       ] (  5/138[=                                       ] (  6/13[==                                      ] (  7/13[==                                      ] (  8/13[==                                      ] (  9/13[==                                      ] ( 10/138[===                                     ] ( 11/1[===                                     ] ( 12/1[===                                     ] ( 13/13[====                                    ] ( 14/13[====                                    ] ( 15/13[====                                    ] ( 16/13[====                                    ] ( 17/138[=====                                   ] ( 18/138[====

In [15]:
with open("man2.md","w",encoding="utf-8") as f:
    f.write(loaded_document)


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()
# embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5",trust_remote_code=True)


d:\Chatbot\parent_retrival\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
# loaded_document
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 3"),
    ("#####", "Header 3"),
    ("######", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header_splits = markdown_splitter.split_text(loaded_document)
md_header_splits

[Document(metadata={'Header 3': 'Executive Summary'}, page_content='❖ The Executive Service Rules first introduced in May, 1981 and amended, as per BOD’s  \napprovals from time to time are currently applicable.  \n❖ On the directions of HR,R&NC / BOD, services of HR Consultant were commissioned  \nto review and assess suitability of existing rules per current market conditions & propose\namendments in policies, practices and document & prepare a new Manual.  \n❖ Financial & Admin. Powers for each Position / Level have also been made an integral  \npart of HR Manual, as per advice of Chairman, HR,R&NC.  \n❖ The Board of Directors approved the HR Manual in its 344[th ] meeting held on 10-06-08.  \nIn addition to amendment in different clauses from time to time, the HR Manual has\nbeen thoroughly reviewed, revised and approved by the Board of Directors in its\nfollowing meetings:  \n- 485[th ]meeting held on 18-03-2017  \n- 587[th ]meeting held on 21-06-2022  \n- 631[st ]meeting held on 0

In [4]:
# Char-level splits
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 250
chunk_overlap = 30
childesplitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)
parentsplitter = RecursiveCharacterTextSplitter(
    chunk_size=4000, chunk_overlap=0
)

# Split
splits = childesplitter.split_documents(md_header_splits)
splits

[Document(metadata={'Header 3': 'Executive Summary'}, page_content='❖ The Executive Service Rules first introduced in May, 1981 and amended, as per BOD’s  \napprovals from time to time are currently applicable.  \n❖ On the directions of HR,R&NC / BOD, services of HR Consultant were commissioned'),
 Document(metadata={'Header 3': 'Executive Summary'}, page_content='to review and assess suitability of existing rules per current market conditions & propose\namendments in policies, practices and document & prepare a new Manual.'),
 Document(metadata={'Header 3': 'Executive Summary'}, page_content='❖ Financial & Admin. Powers for each Position / Level have also been made an integral  \npart of HR Manual, as per advice of Chairman, HR,R&NC.  \n❖ The Board of Directors approved the HR Manual in its 344[th ] meeting held on 10-06-08.'),
 Document(metadata={'Header 3': 'Executive Summary'}, page_content='In addition to amendment in different clauses from time to time, the HR Manual has\nbeen th

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.storage._lc_store import create_kv_docstore
from langchain.storage import LocalFileStore
from langchain_chroma import Chroma

#vector_store
vector_store = Chroma(collection_name="HR_manual",embedding_function=embeddings,persist_directory="./RAG")
# create local_store
fs = LocalFileStore("./store_location")
store = create_kv_docstore(fs)


# define child splitter
child_splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,  # must be less
    chunk_overlap=50,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
    # length_function=len
)

# Define parent splitter
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=4000)


In [6]:
from langchain.retrievers import ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=store,
    child_splitter=childesplitter,
    parent_splitter=parentsplitter,
    # search_kwargs = {"k": 20},
)

In [11]:
len(list(store.yield_keys()))

0

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()
api_key=os.getenv("GOOGLE_API_KEY")
llm=ChatGoogleGenerativeAI(model="gemini-1.5-flash",api_key=api_key)

In [6]:
# loaded_document
from langchain_community.document_loaders import UnstructuredMarkdownLoader
loader = UnstructuredMarkdownLoader("man2.md")
data = loader.load()

In [8]:
retriever.add_documents(md_header_splits)

In [9]:
len(list(store.yield_keys()))

296

In [42]:
ex=llm.invoke(f"you are assistant that is use for query expansion. you need to apply expansion query to the given query and return a single/ to the point expanded query. this is the query:  Business Code of Conduct")


In [43]:
ex.content

'Business ethics and conduct guidelines'

In [44]:
query="maximum age required by grade 1 officer?"
result=retriever.invoke(ex.content.lower())
print(result)
# result=vector_store.similarity_search(query.lower())
llm.invoke(f"You are a factual and precise assistant. Below is trusted context extracted from verified documents. Your task is to answer the query using only the information contained in the context. Do not add any details or assumptions that are not explicitly present in the provided context. If the context does not include the answer, respond with “I don’t know” rather than making up information.Context:{result}\nQuestion:{ex.content}").content



[Document(metadata={'Header 1': '– Legal Name – Home address and telephone number – Marital status – Dependant details – Emergency contact details – Academic/professional qualifications', 'Header 2': '– Any other legitimate deductions.', 'Header 3': 'Business Code of Conduct'}, page_content="**90.00** **Statement of Intent**  \n90.1 SNGPL believes the key to the proper functioning and maintenance of public and\n**stakeholder** confidence in the Company, is that employees perform their duties with\nhonesty and integrity.  \n90.2 SNGPL requires its entire staff both in executive and subordinate cadres to observe the\nhighest ethical standards in the conduct of its business activities to minimize the\nsignificant risk associated with non compliance. This policy is intended to assist SNPGL\nstaff in meeting the standards of professional and personal integrity expected of them.\nSNGPL staff will act with honesty at all times, protecting and safeguarding the reputation\nof the Company. Any c

"SNGPL's Code of Conduct (91.00) sets standards of professional behavior and ethical conduct for all employees.  It emphasizes impartiality in job performance (91.2), avoiding conflicts of interest (91.3), and using common sense in situations not explicitly covered by policy (91.4).  Employees must maintain confidentiality of company and customer information (92.00), both during and after employment, and avoid unintentional disclosure (92.2).  Honesty and integrity are paramount (93.00); employees must not misappropriate funds or property (93.1) and must ensure accuracy in information given to customers (93.1).  Non-compliance may result in disciplinary action or dismissal (91.3)."

In [12]:
query="mobile policy"
result=retriever.invoke(query.lower())
len(result[1].page_content)

3998

In [13]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from langchain_community.document_loaders import UnstructuredMarkdownLoader

# ✅ Download necessary NLTK resources
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

def preprocess_text(text):
    """Cleans and preprocesses text before further use."""
    tokens = word_tokenize(text.lower())  # Lowercasing & tokenization
    tokens = [word for word in tokens if word.isalnum()]  # Remove punctuation
    tokens = [word for word in tokens if word not in stopwords.words("english")]  # Remove stopwords
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatization
    return " ".join(tokens)

# ✅ Load raw document
loader = UnstructuredMarkdownLoader("man.md")
data = loader.load()

# ✅ Extract and preprocess text
raw_texts = [doc.page_content for doc in data]
preprocessed_texts = [preprocess_text(text) for text in raw_texts]

# ✅ Output preprocessed text (for debugging)
for i, text in enumerate(preprocessed_texts):
    print(f"Processed Document {i+1}:")
    print(text[:500])  # Print first 500 chars of each processed document
    print("-" * 80)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\husnain.mahmood\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\husnain.mahmood\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\husnain.mahmood\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


FileNotFoundError: [Errno 2] No such file or directory: 'man.md'